            INDEX TIME (Once)

                PDF
                ↓
                Extract Text
                ↓
                Chunk Text
                ↓
                Generate Embeddings
                ↓
                Store in db - chromadb/or smthng


             QUERY TIME (Every Question)

                User Question
                ↓
                Generate Question Embedding
                ↓
                Similarity Search in db
                ↓
                Retrieve Top-K Chunks
                ↓
                Build Prompt (Context + Question)
                ↓
                LLM
                ↓
                Grounded Answer + Sources

In [1]:
from dotenv import load_dotenv
import os
from retrival import retrieve

from groq import Groq


c:\Users\ASUS\PycharmProjects\Machine-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5816.56it/s]


In [2]:
load_dotenv()
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

In [3]:
def build_prompt(question, retrieved_chunks):

    context = ""

    for chunk in retrieved_chunks:

        context += (
            f"Source: {chunk['source']}\n"
            f"{chunk['text']}\n\n"
        )

    prompt = f"""
            You are a helpful AI assistant.

            Answer Only using the context below.

            If the answer is not present in the context,
            reply exactly:

            "I don't know."

            Context
            --------------------
            {context}
            --------------------

            Question:
            {question}

            Answer:
            """

    return prompt

In [4]:
def generate_answer(prompt):

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content


In [5]:
def rag_query(question):

    chunks = retrieve(question)

    prompt = build_prompt(question, chunks)

    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "sources": list(set(chunk["source"] for chunk in chunks))
    }


In [ ]:
while True:

        question = input("\nAsk a question (type 'exit' to quit): ")

        if question.lower() == "exit":
            break

        result = rag_query(question)

        print("\nAnswer")
        print("=" * 60)
        print(result["answer"])

        print("\nSources")
        print("=" * 60)

        for source in result["sources"]:
            print(f"- {source}")